# Pipeline visualization (post-run)

This notebook **does not** run the CNMF pipeline. Run processing first, for example:

```bash
minian-pipeline -d /path/to/your/folder
```

Then set `DATA_DIR` below to the same folder you passed as `--data` / `-d`.

**On disk** (same layout as `notebooks/pipeline_test.ipynb` / the headless pipeline):

- **Intermediate Zarr** — `{DATA_DIR}/minian_intermediate/` (`varr`, `varr_ref`, `Y_fm_chk`, `Y_hw_chk`, `C_chk`, `A_init`, `C_init`, `sn_spatial`, `YrA`, … depending on run).
- **Merged results** — `{DATA_DIR}/minian/` — final `A`, `C`, `S`, `b`, `f`, `b0`, `c0`, `motion`, `max_proj`, …

Figures mirror the **tutorial / pipeline_test** notebook where the underlying arrays exist on disk. Steps that only keep in-memory objects (`seeds`, `pnr`, `gmm`, per-iteration `A_new` / `C_new` sweeps) are skipped or noted.

Before loading, the notebook calls `require_existing_dirs` so missing Zarr roots fail fast with a clear message.


## Configuration

Match `DATA_DIR` to your pipeline run. Tune `output_size` and `interactive` like `pipeline_test.ipynb` (`interactive=False` skips heavy Panel viewers).


In [ ]:
import os

import holoviews as hv
import numpy as np
import xarray as xr
from bokeh.io import curdoc
from holoviews.operation.datashader import datashade, regrid
from IPython.display import display
from dask.distributed import Client, LocalCluster

from minian.cnmf import compute_AtC
from minian.constants import get_minian_intermediate_path, minian_folder_under
from minian.preprocessing import denoise, remove_background
from minian.utilities import TaskAnnotation, open_minian, require_existing_dirs
from minian.utilities.logger import configure_logging
from minian.visualization import (
    CNMFViewer,
    VArrayViewer,
    generate_videos,
    visualize_motion,
    visualize_preprocess,
)

hv.extension("bokeh")
doc = curdoc()
doc.clear()

DATA_DIR = os.path.abspath("./demo_movies")
INTPATH = get_minian_intermediate_path(DATA_DIR)
MINIAN_PATH = minian_folder_under(DATA_DIR)

subset = dict(frame=slice(0, None))
output_size = 100
interactive = True

for label, p in (
    ("DATA_DIR", DATA_DIR),
    ("INTPATH", INTPATH),
    ("MINIAN_PATH", MINIAN_PATH),
):
    print(f"{label}: {p}")

configure_logging(os.getenv("MINIAN_LOG_LEVEL", "INFO"), force=True)

## Dask (optional)

LocalCluster helps when viewers and `regrid` / `datashade` trigger large `.compute()` calls. Comment out this cell and the teardown cell if you prefer the threaded scheduler only.


In [ ]:
from minian.config import PipelineConfig

_viz_cluster_cfg = PipelineConfig(
    intpath=INTPATH,
    n_workers=4,
    dask_chunk_target_mb=100,
)
n_workers = _viz_cluster_cfg.resolved_n_workers()
threads_per_worker = _viz_cluster_cfg.dask_threads_per_worker
worker_memory_limit = _viz_cluster_cfg.dask_worker_memory
chunk_target_mb = _viz_cluster_cfg.dask_chunk_target_mb

cluster = LocalCluster(
    n_workers=n_workers,
    memory_limit=worker_memory_limit,
    resources={"MEM": 1},
    threads_per_worker=threads_per_worker,
    dashboard_address=":8787",
)
annt_plugin = TaskAnnotation()
cluster.scheduler.add_plugin(annt_plugin)
client = Client(cluster)

print(
    f"Started Dask LocalCluster at {cluster.scheduler.address!r}\n"
    f"  n_workers={n_workers}, memory_limit={worker_memory_limit!r}, "
    f"threads_per_worker={threads_per_worker}, chunk_target_mb={chunk_target_mb}\n"
    f"  dashboard {client.dashboard_link!r}"
)

## Load saved Minian stores

`require_existing_dirs` (from `minian.utilities`) checks that the intermediate and merged folders exist, then `open_minian` merges every `*.zarr` child directory under each path (same as `pipeline_test` after saves).


In [ ]:
require_existing_dirs(
    {
        "intermediate (intpath)": INTPATH,
        "minian output": MINIAN_PATH,
    },
    hint="Run the pipeline first.",
)

ds_int = open_minian(INTPATH)
ds_out = open_minian(MINIAN_PATH)


def _require(ds: xr.Dataset, name: str) -> xr.DataArray:
    if name not in ds.data_vars:
        raise KeyError(
            f"{name!r} not in dataset; have {sorted(map(str, ds.data_vars))}"
        )
    return ds[name]


def _maybe(ds: xr.Dataset, name: str) -> xr.DataArray | None:
    return ds[name] if name in ds.data_vars else None


varr = _require(ds_int, "varr")
varr_ref = _require(ds_int, "varr_ref")
Y_fm_chk = _require(ds_int, "Y_fm_chk")
motion = _require(ds_out, "motion")
max_proj = _require(ds_out, "max_proj")

A = _require(ds_out, "A")
C = _require(ds_out, "C")
S = _require(ds_out, "S")
C_chk = _require(ds_int, "C_chk")

Y_hw_chk = _maybe(ds_int, "Y_hw_chk")
A_init = _maybe(ds_int, "A_init")
C_init = _maybe(ds_int, "C_init")
sn_spatial = _maybe(ds_int, "sn_spatial")
YrA = _maybe(ds_int, "YrA")

b = _maybe(ds_out, "b")
f = _maybe(ds_out, "f")
b0 = _maybe(ds_out, "b0")
c0 = _maybe(ds_out, "c0")

print("intermediate:", sorted(map(str, ds_int.data_vars)))
print("minian out:  ", sorted(map(str, ds_out.data_vars)))

## Raw movie (`varr`)

Same pattern as `pipeline_test`: `hv.output` + optional `VArrayViewer.show()`.


In [ ]:
hv.output(size=output_size)
if interactive:
    display(VArrayViewer(varr, framerate=5, summary=["mean", "max"]).show())

## Raw vs preprocessed (`varr` / `varr_ref`) — side-by-side layout

Matches the tutorial cell that passes two arrays with `layout=True`.


In [ ]:
hv.output(size=int(output_size * 0.7))
if interactive:
    display(
        VArrayViewer(
            [varr.rename("original"), varr_ref.rename("glow_removed")],
            framerate=5,
            summary=None,
            layout=True,
        ).show()
    )

## After denoise + background removal (`varr_ref` only)


In [ ]:
hv.output(size=output_size)
if interactive:
    display(VArrayViewer(varr_ref, framerate=5, summary=["mean", "max"]).show())

## Preprocess parameter exploration (`visualize_preprocess`)

Same calls as `pipeline_test`: one frame from `varr_ref`, sweep **median** `ksize` then **tophat** `wnd`. (This is not the same as comparing two finished arrays — that API takes a single frame + optional `fn` + parameter grids.)


In [ ]:
hv.output(size=int(output_size * 0.6))
if interactive:
    display(
        visualize_preprocess(
            varr_ref.isel(frame=0).compute(),
            denoise,
            method=["median"],
            ksize=[5, 7, 9],
        )
    )

In [ ]:
hv.output(size=int(output_size * 0.6))
if interactive:
    display(
        visualize_preprocess(
            varr_ref.isel(frame=0).compute(),
            remove_background,
            method=["tophat"],
            wnd=[10, 15, 20],
        )
    )

## Motion correction — movies and shifts

`VArrayViewer` with `before_mc` / `after_mc` labels (tutorial), then `visualize_motion`.


In [ ]:
hv.output(size=int(output_size * 0.7))
if interactive:
    display(
        VArrayViewer(
            [varr_ref.rename("before_mc"), Y_fm_chk.rename("after_mc")],
            framerate=5,
            summary=None,
            layout=True,
        ).show()
    )

In [ ]:
hv.output(size=output_size)
visualize_motion(motion)

## Max projections: FOV before MC vs height/width–registered FOV

Uses `regrid` + `hv.Image` like `pipeline_test` (requires `Y_hw_chk` on disk).


In [ ]:
if Y_hw_chk is None:
    print("Y_hw_chk not in intermediate store — skip max-projection comparison.")
else:
    im_opts = dict(
        frame_width=500,
        aspect=varr_ref.sizes["width"] / varr_ref.sizes["height"],
        cmap="Viridis",
        colorbar=True,
        show_title=False,
    )
    hv.output(size=output_size)
    display(
        (
            regrid(
                hv.Image(
                    varr_ref.max("frame").compute().astype(np.float32),
                    ["width", "height"],
                    label="before_mc",
                ).opts(**im_opts)
            )
            + regrid(
                hv.Image(
                    Y_hw_chk.max("frame").compute().astype(np.float32),
                    ["width", "height"],
                    label="after_mc",
                ).opts(**im_opts)
            )
        )
    )

## Saved `max_proj` (used for seeding)

`visualize_seeds` / `visualize_gmm_fit` from the tutorial need in-memory `seeds` / `pnr` / `gmm`, which the headless pipeline does not save.


In [ ]:
hv.output(size=output_size)
regrid(
    hv.Image(
        max_proj.astype(np.float32),
        kdims=["width", "height"],
        label="max_proj",
    ).opts(
        frame_width=500,
        aspect=int(max_proj.sizes["width"]) / int(max_proj.sizes["height"]),
        cmap="Viridis",
        colorbar=True,
    )
)

## Noise estimate (`sn_spatial`)

Saved after `get_noise_fft` in the full pipeline (intermediate store).


In [ ]:
if sn_spatial is None:
    print("sn_spatial not found — skip.")
else:
    hv.output(size=int(output_size * 0.6))
    display(
        regrid(
            hv.Image(
                sn_spatial.astype(np.float32),
                kdims=["width", "height"],
            ).opts(
                frame_width=500,
                aspect=int(sn_spatial.sizes["width"]) / int(sn_spatial.sizes["height"]),
                cmap="Viridis",
                colorbar=True,
            )
        ).relabel("sn_spatial")
    )

## Initial `A_init` / `C_init` and final background `b` / `f`

Tutorial cell combines init spatial/temporal footprints with **initial** background; here we show **saved** `A_init`/`C_init` plus **final** `b`/`f` from the merged minian folder (after full CNMF).


In [ ]:
hv.output(size=int(output_size * 0.55))
if A_init is None or C_init is None:
    print("A_init or C_init missing — skip init footprint panel.")
elif b is None or f is None:
    print("b or f missing on merged dataset — skip background row.")
else:
    im_opts = dict(
        frame_width=500,
        aspect=A_init.sizes["width"] / A_init.sizes["height"],
        cmap="Viridis",
        colorbar=True,
        show_title=False,
    )
    cr_opts = dict(
        frame_width=750,
        aspect=1.5 * A_init.sizes["width"] / A_init.sizes["height"],
        show_title=False,
    )
    display(
        (
            regrid(
                hv.Image(
                    A_init.max("unit_id").rename("A_init").compute().astype(np.float32),
                    kdims=["width", "height"],
                ).opts(**im_opts)
            ).relabel("Initial spatial (max over units)")
            + regrid(
                hv.Image(
                    C_init.rename("C_init").compute().astype(np.float32),
                    kdims=["frame", "unit_id"],
                ).opts(cmap="viridis", colorbar=True, **cr_opts)
            ).relabel("Initial C_init")
            + regrid(
                hv.Image(
                    b.rename("b").compute().astype(np.float32),
                    kdims=["width", "height"],
                ).opts(**im_opts)
            ).relabel("Final background spatial b")
            + datashade(
                hv.Curve(f.rename("f").compute(), kdims=["frame"]), min_alpha=200
            )
            .opts(**cr_opts)
            .relabel("Final background temporal f")
        ).cols(2)
    )

## Final spatial footprints (max projection + occupancy)

Same idea as the “Spatial Footprints Last” panels in `pipeline_test`, using merged `A`.


In [ ]:
hv.output(size=int(output_size * 0.6))
opts = dict(
    plot=dict(
        height=int(A.sizes["height"]), width=int(A.sizes["width"]), colorbar=True
    ),
    style=dict(cmap="Viridis"),
)
display(
    (
        regrid(
            hv.Image(
                A.max("unit_id").compute().astype(np.float32).rename("A"),
                kdims=["width", "height"],
            ).opts(**opts)
        ).relabel("Spatial max over units")
        + regrid(
            hv.Image(
                (A.fillna(0) > 0).sum("unit_id").compute().astype(np.uint8).rename("A"),
                kdims=["width", "height"],
            ).opts(**opts)
        ).relabel("Binary occupancy (sum units)")
    ).cols(2)
)

## Final temporal `C` and `S` heatmaps


In [ ]:
hv.output(size=int(output_size * 0.6))
opts_im = dict(frame_width=500, aspect=2, colorbar=True, cmap="Viridis")
display(
    (
        regrid(
            hv.Image(
                C.compute().astype(np.float32).rename("C"), kdims=["frame", "unit_id"]
            ).opts(**opts_im)
        ).relabel("C")
        + regrid(
            hv.Image(
                S.compute().astype(np.float32).rename("S"), kdims=["frame", "unit_id"]
            ).opts(**opts_im)
        ).relabel("S")
    ).cols(2)
)

## Residual traces `YrA` (sample units)

Saved intermittently on the intermediate path; this plots a few `unit_id` curves (tutorial uses `YrA` inside `visualize_temporal_update`).


In [ ]:
if YrA is None:
    print("YrA not in intermediate store — skip.")
else:
    hv.output(size=int(output_size * 0.6))
    uids = YrA.coords["unit_id"].values[: min(8, YrA.sizes["unit_id"])]
    curves = hv.Overlay()
    for u in uids:
        curves *= hv.Curve(
            YrA.sel(unit_id=u).compute().astype(np.float32),
            kdims=["frame"],
        ).relabel(str(u))
    display(curves.opts(hv.opts.Curve(width=700, height=200)))

## Denoised reconstruction heatmap (`compute_AtC`)

Optional overview of `A @ C` in frame–unit space (same helper as `generate_videos` uses when `AC` is not passed).


In [ ]:
hv.output(size=int(output_size * 0.6))
AC = compute_AtC(A, C_chk)
display(
    regrid(
        hv.Image(
            AC.compute().astype(np.float32),
            kdims=["frame", "unit_id"],
        ).opts(frame_width=600, aspect=2, cmap="Viridis", colorbar=True)
    ).relabel("AtC (using C_chk)")
)

## Summary video (`generate_videos`)

Same call signature as the pipeline / `pipeline_test`.


In [ ]:
generate_videos(varr.sel(subset), Y_fm_chk, A=A, C=C_chk, vpath=DATA_DIR)

## Interactive CNMF explorer (`CNMFViewer`)


In [ ]:
if interactive:
    display(CNMFViewer(A=A, C=C, S=S, org=Y_fm_chk).show())

## Teardown

Closes the Dask cluster if those objects exist (skipped entirely when the cluster cell was not run).


In [ ]:
client_ = globals().get("client")
cluster_ = globals().get("cluster")
if client_ is not None:
    client_.close()
if cluster_ is not None:
    cluster_.close()